In [1]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


In [2]:
import os
import torch
from torchvision import transforms
import timm # provides pretrained models

model = timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, dynamic_img_size=True)
model.to('mps')
# put the model into inference mode; ensure consistent outputs when extracting embeddings
model.eval()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inpl

In [3]:
torch.cuda.is_available() 

False

In [4]:
torch.mps.is_available()

True

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import AutoImageProcessor, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tqdm import tqdm
import numpy as np

# # -------------------------------
# # Configuration
# # -------------------------------
DATA_DIR = "/Users/jk/Documents/Lab2/visium/python_analysis/cpath/visium_patches/55um"
MODEL_NAME = "MahmoodLab/UNI"   # Hugging Face repo
BATCH_SIZE = 32 # number of images processed in parallel
DEVICE = "mps" 

# # -------------------------------
# # Load model & processor
# # -------------------------------
# processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
# # model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
# # model.eval()  # Freeze encoder

# # -------------------------------
# # Image preprocessing
# # -------------------------------
transform = transforms.Compose(
    [
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)

# -------------------------------
# Create datasets
# -------------------------------
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Extract class names
class_names = dataset.classes
print("Classes:", class_names)

# -------------------------------
# Extract embeddings
# -------------------------------
features, labels = [], []
with torch.no_grad(): # disable gradient computation
    for images, lbls in tqdm(loader, desc="Extracting UNI embeddings"):
        images = images.to('mps')
        outputs = model(images)  # directly use your timm model
        # UNI uses CLS token output
        if isinstance(outputs, (list, tuple)):
            embeddings = outputs[0]
        else:
            embeddings = outputs
        features.append(embeddings.cpu().numpy())
        labels.append(lbls.numpy())

features = np.concatenate(features)
labels = np.concatenate(labels)

# -------------------------------
# Split train/validation
# -------------------------------
# from sklearn.model_selection import train_test_split
# X_train, X_val, y_train, y_val = train_test_split(features, labels, test_size=0.2, stratify=labels, random_state=42)

# # -------------------------------
# # Train simple classifier
# # -------------------------------
# clf = LogisticRegression(max_iter=500, multi_class='multinomial')
# clf.fit(X_train, y_train)

# # -------------------------------
# # Evaluate
# # -------------------------------
# y_pred = clf.predict(X_val)
# print(classification_report(y_val, y_pred, target_names=class_names))


Classes: ['cluster0', 'cluster1', 'cluster2', 'cluster3', 'cluster4', 'cluster5']


Extracting UNI embeddings: 100%|██████████| 3140/3140 [32:52<00:00,  1.59it/s]


In [6]:
# -------------------------------
# Split train/validation
# -------------------------------
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(features, labels, test_size=0.2, stratify=labels, random_state=42)

# -------------------------------
# Train simple classifier
# -------------------------------
clf = LogisticRegression(max_iter=500, multi_class='multinomial')
clf.fit(X_train, y_train)

# -------------------------------
# Evaluate
# -------------------------------
y_pred = clf.predict(X_val)
print(classification_report(y_val, y_pred, target_names=class_names))

/Users/jk/miniconda3/envs/uni/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

    cluster0       0.47      0.48      0.48      3941
    cluster1       0.44      0.46      0.45      3900
    cluster2       0.51      0.59      0.55      3782
    cluster3       0.39      0.29      0.33      2763
    cluster4       0.50      0.45      0.47      2868
    cluster5       0.52      0.56      0.54      2841

    accuracy                           0.48     20095
   macro avg       0.47      0.47      0.47     20095
weighted avg       0.47      0.48      0.47     20095



/Users/jk/miniconda3/envs/uni/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
